In [ ]:
# validate.ipynb
import os
import mne
import numpy as np
import joblib
from scipy.signal import welch, spectrogram
from scipy.stats import entropy
from nolds import lyap_r
from sklearn.metrics import accuracy_score, recall_score, precision_score, confusion_matrix
from joblib import Parallel, delayed

mne.set_log_level("WARNING")

# Set DATA_DIR to the folder containing your .edf files
DATA_DIR = r"C:\Users\aaww8\OneDrive\바탕 화면\phython workspace\.vscode\chb-mit-scalp-eeg-database-1.0.0"
# GitHub: DATA_DIR = "data"

# Ground truth
seizure_start = 1467
seizure_end   = 1494

# Evaluation window
test_start = 1360
test_end   = 1560

edf_file = os.path.join(DATA_DIR, "chb01_04.edf")
raw_new  = mne.io.read_raw_edf(edf_file, preload=True, verbose=False)
raw_new.crop(tmin=test_start, tmax=test_end)

channels = [
    "FP1-F7", "F7-T7",   "T7-P7",   "P7-O1",
    "FP1-F3", "F3-C3",   "C3-P3",   "P3-O1",
    "FP2-F4", "F4-C4",   "C4-P4",   "P4-O2",
    "FP2-F8", "F8-T8",   "T8-P8-1", "T8-P8-0",
    "P8-O2",  "FZ-CZ",   "CZ-PZ"
]
raw_new.pick_channels(channels)
raw_new.filter(l_freq=0.5, h_freq=40, verbose=False)
raw_new.resample(sfreq=128, verbose=False)
raw_new.set_eeg_reference("average", projection=True, verbose=False)
raw_new.apply_proj()

epochs_new     = mne.make_fixed_length_epochs(raw_new, duration=2.0, preload=True, verbose=False)
new_data_array = epochs_new.get_data()
print(f"Validation data shape: {new_data_array.shape}")

# y_true from physician annotations
y_true = []
for i in range(len(new_data_array)):
    epoch_start = test_start + (i * 2)
    epoch_end   = epoch_start + 2
    y_true.append(1 if epoch_end > seizure_start and epoch_start < seizure_end else 0)
y_true = np.array(y_true)
print(f"Seizure epochs: {np.sum(y_true)}, Normal epochs: {np.sum(y_true == 0)}")

# Feature extraction 
feature_ptp = np.ptp(new_data_array, axis=2)
feature_var = np.var(new_data_array, axis=2)

freqs, psd  = welch(new_data_array, fs=128, nperseg=256, axis=-1)
delta_power = np.mean(psd[:, :, (freqs >= 0.5) & (freqs < 4)],  axis=-1)
theta_power = np.mean(psd[:, :, (freqs >= 4)   & (freqs < 8)],  axis=-1)
alpha_power = np.mean(psd[:, :, (freqs >= 8)   & (freqs < 12)], axis=-1)
beta_power  = np.mean(psd[:, :, (freqs >= 12)  & (freqs < 30)], axis=-1)

n_epochs, n_channels, n_times = new_data_array.shape

def spectral_entropy(signal, sf, nperseg=256, normalize=True):
    _, psd_s = welch(signal, sf, nperseg=nperseg)
    if normalize:
        psd_s = psd_s / np.sum(psd_s)
    return entropy(psd_s, base=2)

spectral_entropy_features = np.apply_along_axis(
    spectral_entropy, 2, new_data_array, sf=128
)

def lyapunov_exponent(signal, emb_dim=10, lag=None):
    if lag is None:
        lag = emb_dim // 2
    return lyap_r(signal, emb_dim=emb_dim, lag=lag)

signals  = new_data_array.reshape(-1, n_times)
lya_flat = Parallel(n_jobs=-1)(delayed(lyapunov_exponent)(s) for s in signals)
lyapunov_features = np.array(lya_flat).reshape(n_epochs, n_channels)

freqs_spec, times_spec, spec = spectrogram(new_data_array, fs=128, nperseg=64, axis=-1)
delta_power_spec_mean = np.mean(np.mean(spec[:, :, (freqs_spec >= 0.5) & (freqs_spec < 4)],  axis=-1), axis=2)
theta_power_spec_mean = np.mean(np.mean(spec[:, :, (freqs_spec >= 4)   & (freqs_spec < 8)],  axis=-1), axis=2)
alpha_power_spec_mean = np.mean(np.mean(spec[:, :, (freqs_spec >= 8)   & (freqs_spec < 12)], axis=-1), axis=2)
beta_power_spec_mean  = np.mean(np.mean(spec[:, :, (freqs_spec >= 12)  & (freqs_spec < 30)], axis=-1), axis=2)

features_new = np.concatenate((
    feature_ptp, feature_var,
    delta_power, theta_power, alpha_power, beta_power,
    delta_power_spec_mean, theta_power_spec_mean, alpha_power_spec_mean, beta_power_spec_mean,
    spectral_entropy_features, lyapunov_features
), axis=1)

# Predict & evaluate
loaded_model   = joblib.load("xgb_spike_detector_15percent.pkl")
loaded_indices = joblib.load("top_30_features_indices.pkl")

features_new_selected = features_new[:, loaded_indices]
probabilities         = loaded_model.predict_proba(features_new_selected)[:, 1]
y_pred                = (probabilities >= 0.15).astype(int)

acc       = accuracy_score(y_true, y_pred)
recall    = recall_score(y_true, y_pred, zero_division=0)
precision = precision_score(y_true, y_pred, zero_division=0)
cm        = confusion_matrix(y_true, y_pred)


print("\n[Validation Results — chb01_04]")

print(f"Evaluation window : {test_start}–{test_end} s ({len(y_true)} epochs)")
print(f"Threshold         : 15%")
print(f"Recall            : {recall*100:.1f}%")
print(f"Precision         : {precision*100:.1f}%")
print(f"Accuracy          : {acc*100:.1f}%")
print(f"TN: {cm[0,0]}  FP: {cm[0,1]}")
print(f"FN: {cm[1,0]}  TP: {cm[1,1]}")

# Threshold grid search (15%–35%)
print("\nThreshold sweep — Recall 100% lower bound search:")
best_threshold = 0.15
best_precision = 0.0

for thresh in np.arange(0.15, 0.36, 0.01):
    y_pred_temp = (probabilities >= thresh).astype(int)
    r = recall_score(y_true, y_pred_temp, zero_division=0)
    p = precision_score(y_true, y_pred_temp, zero_division=0)
    print(f"  {thresh*100:.0f}%  Recall: {r*100:.1f}%  Precision: {p*100:.1f}%")
    if r == 1.0 and p > best_precision:
        best_precision = p
        best_threshold = thresh

print(f"\nOptimal threshold (Recall=100%): {best_threshold*100:.0f}%  "
      f"Precision: {best_precision*100:.1f}%")
